In [1]:
# Constantes de ronda oficiales para Keccak-f (24 rondas)
RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008
]

# Matriz de rotaciones fijas para el paso Rho
ROTATION_OFFSETS = [
    [0,  36,  3,  41, 18],
    [1,  44, 10,  45,  2],
    [62,  6, 43,  15, 61],
    [28, 55, 25,  21, 56],
    [27, 20, 39,   8, 14]
]

def ROTL64(x, n):
    """Rotación de bits circular para un entero de 64 bits."""
    return ((x << (n % 64)) & 0xFFFFFFFFFFFFFFFF) | (x >> (64 - (n % 64)))

def keccak_f1600(state):
    """Permutación nativa Keccak-f sobre una matriz de 5x5 enteros de 64 bits."""
    for round_idx in range(24):
        # 1. Paso Theta (θ)
        C = [0] * 5
        for x in range(5):
            C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]

        D = [0] * 5
        for x in range(5):
            D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)

        for x in range(5):
            for y in range(5):
                state[x][y] ^= D[x]

        # 2. Pasos Rho (ρ) y Pi (π) combinados
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[y][(2 * x + 3 * y) % 5] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
        state = next_state

        # 3. Paso Chi (χ)
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
        state = next_state

        # 4. Paso Iota (ι) - Afecta únicamente al carril de origen (0,0)
        state[0][0] ^= RC[round_idx]

    return state

def sha3_256_fips(message: bytes) -> bytes:
    """Implementación exacta de SHA3-256 según la especificación estándar FIPS 202."""
    rate_bytes = 136  # r = 1088 bits para SHA3-256
    state = [[0] * 5 for _ in range(5)]

    # Construcción matemática del relleno (Padding pad10*1)
    # Se concatena el sufijo del dominio SHA-3 (b'01') junto al bit de inicio b'1' -> 0x06
    padded = bytearray(message)
    padded.append(0x06)

    # Rellenar con ceros intermedios hasta completar el bloque
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)

    # El bit final del padding pad10*1 se activa en el bit más significativo
    padded[-1] |= 0x80

    # Fase de Absorción
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600(state)

    # Fase de Exprimido (Squeezing) - Extraemos 32 bytes (256 bits)
    output = bytearray()
    for i in range(4):  # 4 carriles de 8 bytes = 32 bytes
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


if __name__ == "__main__":
    import hashlib

    casos_prueba = [
        b"",
        b"abc",
        b"El cifrado Keccak cumple con el estandar FIPS 202 sin problemas de compatibilidad.",
        b"a" * 135,  # Frontera de bloque - 1 byte
        b"a" * 136,  # Frontera exacta de un bloque
        b"a" * 137   # Frontera de bloque + 1 byte
    ]

    print("--- VERIFICACIÓN FIPS 202 ---")
    for msg in casos_prueba:
        hash_propio = sha3_256_fips(msg).hex()
        hash_oficial = hashlib.sha3_256(msg).hexdigest()

        print(f"Mensaje len({len(msg)}): {msg[:20]}...")
        print(f"  > Propio:  {hash_propio}")
        print(f"  > Oficial: {hash_oficial}")

        assert hash_propio == hash_oficial, f"¡Error : {msg}!"

    print("\n¡hash  coinciden  con el estándar FIPS.")

--- VERIFICACIÓN FIPS 202 ---
Mensaje len(0): b''...
  > Propio:  a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
  > Oficial: a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
Mensaje len(3): b'abc'...
  > Propio:  3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
  > Oficial: 3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
Mensaje len(82): b'El cifrado Keccak cu'...
  > Propio:  75b132c9693574d34dc6c0f87c9b5a5ed381f43d8e80d656698cb33daeeb99a6
  > Oficial: 75b132c9693574d34dc6c0f87c9b5a5ed381f43d8e80d656698cb33daeeb99a6
Mensaje len(135): b'aaaaaaaaaaaaaaaaaaaa'...
  > Propio:  8094bb53c44cfb1e67b7c30447f9a1c33696d2463ecc1d9c92538913392843c9
  > Oficial: 8094bb53c44cfb1e67b7c30447f9a1c33696d2463ecc1d9c92538913392843c9
Mensaje len(136): b'aaaaaaaaaaaaaaaaaaaa'...
  > Propio:  3fc5559f14db8e453a0a3091edbd2bc25e11528d81c66fa570a4efdcc2695ee1
  > Oficial: 3fc5559f14db8e453a0a3091edbd2bc25e11528d81c66fa570a4efdcc2695ee1
Mensaje

In [2]:
# Constantes de ronda para Keccak-f (24 rondas)
RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008
]

# Desplazamientos de rotación específicos para cada celda de la matriz 5x5
ROTATION_OFFSETS = [
    [0, 36, 3, 41, 18],
    [1, 44, 10, 45, 2],
    [62, 6, 43, 15, 61],
    [28, 55, 25, 21, 56],
    [27, 20, 39, 8, 14]
]

def ROTL64(x, n):
    """Rotación circular a la izquierda de 64 bits."""
    return ((x << (n % 64)) & 0xFFFFFFFFFFFFFFFF) | (x >> (64 - (n % 64)))

def keccak_f1600(state):
    """Ejecuta las 24 rondas sobre la matriz de estado."""
    for round_idx in range(24):
        # 1. Paso Theta (θ)
        C = [0] * 5
        for x in range(5):
            C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]

        D = [0] * 5
        for x in range(5):
            D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)

        for x in range(5):
            for y in range(5):
                state[x][y] ^= D[x]

        # 2. Pasos Rho (ρ) y Pi (π) combinados
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[y][(2 * x + 3 * y) % 5] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
        state = next_state

        # 3. Paso Chi (χ)
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
        state = next_state

        # 4. Paso Iota (ι) afecta sólo a la celda de origen [0][0]
        state[0][0] ^= RC[round_idx]

    return state

def sha3_256(message: bytes) -> bytes:
    """Función de Esponja SHA3-256"""
    rate_bytes = 136  # r = 1088 bits para SHA3-256
    state = [[0] * 5 for _ in range(5)]

    # Relleno oficial pad10*1 junto al dominio b'01' de SHA-3
    padded = bytearray(message)
    padded.append(0x06)
    while (len(padded) % rate_bytes) != (rate_bytes - 1):
        padded.append(0x00)
    padded.append(0x80)

    # Fase de Absorción
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600(state)

    # Fase de Exprimido (Squeezing) - Extraer los 32 bytes del Hash
    output = bytearray()
    for i in range(4):  # 4 palabras * 8 bytes = 32 bytes (256 bits)
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


intentos_fallidos = 0

def registrar_intento_fallido():
    """Incrementa el contador global de intentos fallidos."""
    global intentos_fallidos
    intentos_fallidos += 1


# Pasos individuales de Keccak SEPARADOS para poder reordenarlos dinámicamente.
def theta(state):
    """Paso Theta (θ): difusión de columnas mediante XOR."""
    C = [0] * 5
    for x in range(5):
        C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]
    D = [0] * 5
    for x in range(5):
        D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)
    for x in range(5):
        for y in range(5):
            state[x][y] ^= D[x]
    return state

def rho(state):
    """Paso Rho (ρ): solo rotaciones de cada carril, sin reubicar (separado de Pi)."""
    for x in range(5):
        for y in range(5):
            state[x][y] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
    return state

def pi(state):
    """Paso Pi (π): solo reubicación/permutación de las celdas (separado de Rho)."""
    next_state = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            next_state[y][(2 * x + 3 * y) % 5] = state[x][y]
    return next_state

def chi(state):
    """Paso Chi (χ): no linealidad mediante AND/NOT por filas."""
    next_state = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
    return next_state

def sigma(state, intentos_fallidos):
    """Transformación experimental dependiente de intentos_fallidos.
    Aplica un XOR y una pequeña rotación de bits para introducir variabilidad
    dinámica adicional. """
    for x in range(5):
        for y in range(5):
            # XOR dependiente del contexto (intentos_fallidos) y de la posición
            state[x][y] ^= ((intentos_fallidos << (x + y)) & 0xFFFFFFFFFFFFFFFF)
            # Pequeña rotación adicional dependiente de intentos_fallidos
            state[x][y] = ROTL64(state[x][y], (intentos_fallidos + x + y) % 64)
    return state

def iota(state, round_idx, intentos_fallidos):
    """Paso Iota (ι) con constante de ronda dependiente del contexto.
    Ahora la constante RC depende de intentos_fallidos, lo que busca romper
    parcialmente la predictibilidad estática del hash."""
    RC_modificada = RC[round_idx] ^ intentos_fallidos
    state[0][0] ^= RC_modificada
    return state


def keccak_f1600_variante(state):
    """Permutación Keccak-f adaptativa controlada por intentos_fallidos."""
    # A mayor número de intentos, el número de rondas se acerca progresivamente a 24
    rondas = min(24, 8 + intentos_fallidos)
    for round_idx in range(rondas):
        # Se altera SOLO el pipeline de ejecución para demostrar experimentalmente
        # la NO CONMUTATIVIDAD de las transformaciones de Keccak.
        if intentos_fallidos < 3:
            # Orden original: theta -> rho -> pi -> chi -> iota
            state = theta(state)
            state = rho(state)
            state = pi(state)
            state = chi(state)
            state = iota(state, round_idx, intentos_fallidos)
        else:
            # Orden experimental: theta -> rho -> chi -> pi -> sigma -> iota
            state = theta(state)
            state = rho(state)
            state = chi(state)
            state = pi(state)
            state = sigma(state, intentos_fallidos)
            state = iota(state, round_idx, intentos_fallidos)
    return state


def sha3_256_variante(message: bytes) -> bytes:
    """Variante experimental de SHA3-256 que usa keccak_f1600_variante.
    La sponge construction se mantiene idéntica a la original."""
    rate_bytes = 136  # r = 1088 bits para SHA3-256
    state = [[0] * 5 for _ in range(5)]

    padded = bytearray(message)
    padded.append(0x06)
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)
    padded[-1] |= 0x80

    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600_variante(state)

    output = bytearray()
    for i in range(4):
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


if __name__ == "__main__":
    # Misma entrada de texto para ambos cálculos
    mensaje = b"password123"

    # 1) Hash original Keccak/SHA3-256 estándar
    hash_original = sha3_256_fips(mensaje).hex()

    # Simulamos varios intentos fallidos para activar el orden experimental
    # (intentos_fallidos >= 3) y las rondas/constantes dinámicas.
    for _ in range(5):
        registrar_intento_fallido()

    # 2) Hash de la variante adaptativa
    hash_variante = sha3_256_variante(mensaje).hex()

    print("\n--- COMPARACIÓN EXPERIMENTAL (NO CONMUTATIVIDAD) ---")
    print(f"intentos_fallidos = {intentos_fallidos}")
    print(f"Hash original:  {hash_original}")
    print(f"Hash variante:  {hash_variante}")
    # Los hashes son distintos: cambiar el orden de composición de los pasos
    # altera el resultado. Esto es un contraejemplo práctico de NO CONMUTATIVIDAD.
    print("Los hashes son DISTINTOS -> cambiar el orden de composición altera el resultado.")


--- COMPARACIÓN EXPERIMENTAL (NO CONMUTATIVIDAD) ---
intentos_fallidos = 5
Hash original:  ab3fe4003f14e3ef573417f95e47d4985c482eadd139c08b3758eeae7cc60b9d
Hash variante:  a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33
Los hashes son DISTINTOS -> cambiar el orden de composición altera el resultado.
